# Notebook 05: Campaign ROI Simulation

**Purpose:**
- Assign a targeted campaign to every customer based on RFM segment
- Simulate cost vs CLV uplift per segment
- Quantify CAC reduction vs a flat blanket campaign

**Key result:**
> Targeted CLV-based campaigns reduce customer acquisition costs by **13.6%**  
> (Blanket £12/customer → Targeted £{avg}/customer, total: £52,008 → £44,920)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'sans-serif'
})
COLORS = ['#2563EB', '#16A34A', '#DC2626', '#D97706', '#7C3AED',
          '#0891B2', '#DB2777', '#059669', '#64748B', '#B45309']

os.makedirs('../exports', exist_ok=True)
print('Imports loaded.')

In [ ]:
# ── Load master CLV dataset ────────────────────────────
master = pd.read_csv('../exports/customer_clv_master.csv')
print(f'Customers loaded: {len(master):,}')
master.head(3)

In [ ]:
# ── Campaign Strategy Matrix ───────────────────────────
CAMPAIGN_MATRIX = {
    'Champions':           {'name': 'VIP Early Access',           'cost_per_cust': 15, 'uplift_pct': 0.20},
    'Loyal Customers':     {'name': 'Loyalty Rewards Upsell',     'cost_per_cust': 10, 'uplift_pct': 0.15},
    'New Customers':       {'name': 'Onboarding Series',          'cost_per_cust': 8,  'uplift_pct': 0.25},
    'Potential Loyalists': {'name': 'Membership Offer',           'cost_per_cust': 12, 'uplift_pct': 0.18},
    'Promising':           {'name': 'Cross-Sell Campaign',        'cost_per_cust': 9,  'uplift_pct': 0.12},
    'Need Attention':      {'name': 'Personalized Re-Engagement', 'cost_per_cust': 11, 'uplift_pct': 0.10},
    'At Risk':             {'name': 'Win-Back Discount',          'cost_per_cust': 14, 'uplift_pct': 0.08},
    'Cannot Lose Them':    {'name': 'Priority Win-Back',          'cost_per_cust': 18, 'uplift_pct': 0.12},
    'Lost':                {'name': 'Re-Activation Email',        'cost_per_cust': 5,  'uplift_pct': 0.03},
    'Hibernating':         {'name': 'General Newsletter',         'cost_per_cust': 3,  'uplift_pct': 0.02},
}

print('Campaign matrix defined:')
for seg, cfg in CAMPAIGN_MATRIX.items():
    print(f"  {seg:<22} | {cfg['name']:<30} | "
          f"£{cfg['cost_per_cust']:>3}/cust | uplift {cfg['uplift_pct']*100:.0f}%")

In [ ]:
# ── Build campaign_targets DataFrame ──────────────────
campaign_targets_df = master[['customer_id', 'rfm_segment', 'clv_segment', 'clv_12m']].copy()

campaign_targets_df['campaign_name'] = campaign_targets_df['rfm_segment'].map(
    lambda s: CAMPAIGN_MATRIX.get(s, {'name': 'General Newsletter'})['name']
)
campaign_targets_df['campaign_cost'] = campaign_targets_df['rfm_segment'].map(
    lambda s: CAMPAIGN_MATRIX.get(s, {'cost_per_cust': 3})['cost_per_cust']
)
uplift_map = {seg: v['uplift_pct'] for seg, v in CAMPAIGN_MATRIX.items()}
campaign_targets_df['expected_revenue'] = (
    campaign_targets_df['clv_12m'] *
    campaign_targets_df['rfm_segment'].map(uplift_map).fillna(0.02)
).round(2)

campaign_targets_df['priority_tier'] = campaign_targets_df['clv_segment'].map(
    {'High': 'High', 'Medium': 'Medium', 'Low': 'Low'}
).fillna('Low')

print(f'Campaign targets built: {len(campaign_targets_df):,} customers')
print(campaign_targets_df.groupby('rfm_segment')[['campaign_cost']].mean().round(2))

In [ ]:
# ── campaign_roi DataFrame ─────────────────────────────
campaign_roi_df = (
    campaign_targets_df
    .groupby(['rfm_segment', 'campaign_name'])
    .agg(
        customers      = ('customer_id',      'count'),
        total_cost     = ('campaign_cost',     'sum'),
        avg_clv        = ('clv_12m',           'mean'),
        total_expected = ('expected_revenue',  'sum')
    )
    .reset_index()
)
campaign_roi_df['roi_pct'] = (
    (campaign_roi_df['total_expected'] - campaign_roi_df['total_cost']) /
    campaign_roi_df['total_cost'] * 100
).round(1)

campaign_roi_df = campaign_roi_df.sort_values('roi_pct', ascending=False)
print('Campaign ROI by segment:')
print(campaign_roi_df[['rfm_segment', 'campaign_name', 'customers',
                        'total_cost', 'avg_clv', 'total_expected', 'roi_pct']].to_string(index=False))

In [ ]:
# ── CAC Reduction Calculation ─────────────────────────
BLANKET_COST_PER_CUST = 12  # generic campaign: same cost for all

blanket_total  = len(master) * BLANKET_COST_PER_CUST
targeted_total = campaign_targets_df['campaign_cost'].sum()
cac_reduction  = (blanket_total - targeted_total) / blanket_total * 100

print(f'Blanket campaign total cost (£{BLANKET_COST_PER_CUST}/customer): £{blanket_total:,}')
print(f'Targeted campaign total cost:                         £{targeted_total:,}')
print(f'CAC reduction (targeted vs blanket):                  {cac_reduction:.1f}%')

In [ ]:
# ── Chart 1: Campaign ROI by Segment ─────────────────
roi_plot = campaign_roi_df.sort_values('roi_pct', ascending=True).copy()

def roi_color(v):
    if v >= 200:
        return '#16A34A'   # green
    elif v >= 100:
        return '#D97706'   # amber
    else:
        return '#DC2626'   # red

bar_colors = [roi_color(v) for v in roi_plot['roi_pct']]

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(roi_plot['rfm_segment'], roi_plot['roi_pct'],
               color=bar_colors, alpha=0.85)
for bar, val in zip(bars, roi_plot['roi_pct']):
    ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height() / 2,
            f'{val:.0f}%', va='center', fontsize=9)
ax.axvline(100, color='gray', linestyle='--', linewidth=1, label='Break-even (100%)')
ax.axvline(200, color='green', linestyle=':', linewidth=1, label='High ROI (200%)')
ax.set_title('Campaign ROI by RFM Segment\n(green >200% | amber 100-200% | red <100%)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('ROI (%)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('../exports/campaign_roi.png', bbox_inches='tight')
plt.show()
print('Chart 1 saved: ../exports/campaign_roi.png')

In [ ]:
# ── Chart 2: Targeted vs Blanket Cost Comparison ──────
# Per-segment breakdown: targeted cost vs what blanket would cost
seg_cost = campaign_roi_df[['rfm_segment', 'customers', 'total_cost']].copy()
seg_cost['blanket_cost'] = seg_cost['customers'] * BLANKET_COST_PER_CUST
seg_cost = seg_cost.sort_values('blanket_cost', ascending=False)

x = np.arange(len(seg_cost))
w = 0.35

fig, ax = plt.subplots(figsize=(14, 6))
bars1 = ax.bar(x - w/2, seg_cost['blanket_cost'], w,
               label=f'Blanket (£{BLANKET_COST_PER_CUST}/cust)', color=COLORS[2], alpha=0.80)
bars2 = ax.bar(x + w/2, seg_cost['total_cost'],   w,
               label='Targeted (CLV-based)',       color=COLORS[1], alpha=0.80)

ax.set_xticks(x)
ax.set_xticklabels(seg_cost['rfm_segment'], rotation=30, ha='right', fontsize=9)
ax.set_title('Campaign Spend: Targeted vs Blanket by Segment', fontsize=14, fontweight='bold')
ax.set_ylabel('Total Cost (£)')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f'£{v:,.0f}'))
ax.legend()

# Totals annotation
ax.text(0.98, 0.96,
        f'Blanket total: £{blanket_total:,}\nTargeted total: £{targeted_total:,}\nSaving: {cac_reduction:.1f}%',
        transform=ax.transAxes, ha='right', va='top',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='lightyellow', edgecolor='gray'),
        fontsize=9)

plt.tight_layout()
plt.savefig('../exports/cac_comparison.png', bbox_inches='tight')
plt.show()
print('Chart 2 saved: ../exports/cac_comparison.png')

In [ ]:
# ── Chart 3: Priority Tier Distribution (Donut) ───────
tier_counts = campaign_targets_df['priority_tier'].value_counts()
tier_order  = ['High', 'Medium', 'Low']
tier_counts = tier_counts.reindex([t for t in tier_order if t in tier_counts.index])

fig, ax = plt.subplots(figsize=(7, 7))
wedges, texts, autotexts = ax.pie(
    tier_counts,
    labels=tier_counts.index,
    autopct='%1.1f%%',
    colors=[COLORS[1], COLORS[3], COLORS[2]],
    startangle=90,
    pctdistance=0.82,
    wedgeprops=dict(width=0.6)
)
for at in autotexts:
    at.set_fontsize(11)
ax.set_title('Campaign Priority Tier Distribution\n(High / Medium / Low CLV)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../exports/priority_tiers.png', bbox_inches='tight')
plt.show()
print('Chart 3 saved: ../exports/priority_tiers.png')

In [ ]:
# ── Save CSV exports ───────────────────────────────────
campaign_roi_out = campaign_roi_df[[
    'rfm_segment', 'campaign_name', 'customers',
    'total_cost', 'avg_clv', 'total_expected', 'roi_pct'
]]
campaign_roi_out.to_csv('../exports/campaign_roi.csv', index=False)

campaign_targets_out = campaign_targets_df[[
    'customer_id', 'rfm_segment', 'clv_segment', 'campaign_name',
    'campaign_cost', 'clv_12m', 'expected_revenue', 'priority_tier'
]]
campaign_targets_out.to_csv('../exports/campaign_targets.csv', index=False)

total_incremental = campaign_targets_df['expected_revenue'].sum()

print('Exports saved:')
print('  ../exports/campaign_roi.csv')
print('  ../exports/campaign_targets.csv')
print()
print(f'Campaign simulation complete | CAC reduction: {cac_reduction:.1f}% | '
      f'Total projected incremental revenue: £{total_incremental:,.2f}')